In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import tensorflow as tf
import numpy as np
from scipy.io.wavfile import write
import pickle
import librosa

2025-05-17 11:50:08.960361: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-17 11:50:09.111049: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747453809.164669  945361 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747453809.182257  945361 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1747453809.307138  945361 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
# emo = 'disgust'
# Load model
with open('results/CNN_model.json', 'r') as json_file:
    loaded_model_json = json_file.read()
loaded_model = tf.keras.models.model_from_json(loaded_model_json)
loaded_model.load_weights("results/best_model.weights.h5")
loaded_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Load scaler & encoder
with open("results/scaler.pickle", "rb") as f:
    scaler = pickle.load(f)
with open("results/encoder.pickle", "rb") as f:
    encoder = pickle.load(f)


# target_class = np.where(encoder.categories_[0] == f"{emo}")[0][0]
# print(target_class)
# # Initialize synthetic feature input
# x = tf.Variable(tf.random.normal(shape=(1, 2376, 1), stddev=0.5), trainable=True)

# # Optimization loop
# optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
# for step in range(10000):
#     with tf.GradientTape() as tape:
#         prediction = loaded_model(x, training=False)
# #         loss = loss = tf.abs(target_class - prediction[0, target_class])
#         loss = -tf.math.log(prediction[0, target_class])
#     grads = tape.gradient(loss, x)
#     optimizer.apply_gradients([(grads, x)])

#     # Keep values within plausible standardized range
#     x.assign(tf.clip_by_value(x, -3.0, 3.0))

#     if step % 100 == 0:
#         print(f"Step {step}: {emo} score = {loss.numpy():.4f}")

2025-05-17 11:50:15.859031: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-05-17 11:50:15.859050: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:167] env: CUDA_VISIBLE_DEVICES="-1"
2025-05-17 11:50:15.859055: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:170] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
2025-05-17 11:50:15.859057: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:178] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2025-05-17 11:50:15.859060: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:183] retrieving CUDA diagnostic information for host: aegis-Thin-GF63-12UCX
2025-05-17 11:50:15.859061: I external/local_xla/xla/stream_execu

In [3]:
# # Extract and inverse-scale the optimized features
# features_opt = x.numpy().squeeze()
# features_unscaled = scaler.inverse_transform(features_opt.reshape(1, -1)).squeeze()

# print(features_opt.shape)
# print(features_unscaled.shape)

# # Extract MFCC portion
# mfcc_part = features_unscaled[216:]  # skip ZCR and RMSE
# mfcc_frames = mfcc_part.reshape(108, 20).T  # shape: (n_mfcc, t)

# # Invert to waveform
# audio = librosa.feature.inverse.mfcc_to_audio(mfcc_frames, sr=22050)

# # Save audio
# audio_pcm = (audio * 32767).astype(np.int16)
# write(f"{emo}_generated.wav", 22050, audio_pcm)
# print(f"✅ Saved: {emo}_generated.wav")

In [4]:
# # import numpy as np
# # import scipy.signal
# import soundfile as sf

# # Reuse parameters
# sr = 22050
# duration = 2.0
# n_samples = int(sr * duration)
# t = np.linspace(0, duration, n_samples)

# # Create a synthetic signal that approximates the desired RMSE and ZCR
# # Base signal: modulated sine wave to simulate voiced-like sound
# freq = 220  # Fundamental frequency
# signal = 0.1 * np.sin(2 * np.pi * freq * t)

# # Add noise to increase ZCR and introduce variation in RMSE
# noise = np.random.normal(0, 0.05, size=signal.shape)
# modulated_signal = signal + noise

# # Normalize to simulate desired RMSE mean and std
# current_rmse = np.sqrt(np.mean(modulated_signal**2))
# target_rmse = 0.0692
# scaling_factor = target_rmse / current_rmse
# scaled_signal = modulated_signal * scaling_factor

# # Save audio
# output_path = "angry_audio.wav"
# sf.write(output_path, scaled_signal, sr)

# output_path


In [5]:
emotion_data = {
    'ANGRY': {
        'ZCR_mean': 0.0846,
        'ZCR_std': 0.0296,
        'RMSE_mean': 0.0692,
        'RMSE_std': 0.0478,
        'MFCC_means': [
            -298.124, 121.593, -7.995, 41.206, -17.941, 
            9.999, -15.744, 4.588, -15.002, 1.481, 
            -3.137, -5.640, 3.305, -9.326, 2.893, 
            -9.847, 0.812, -8.127, 0.116, -3.868
        ],
        'MFCC_stds': [
            57.716, 17.098, 15.712, 11.359, 9.764,
            10.442, 7.793, 5.566, 4.787, 4.461,
            4.062, 3.706, 3.975, 3.546, 3.853,
            4.285, 4.058, 3.810, 3.839, 3.803
        ]
    },
    'DISGUST': {
        'ZCR_mean': 0.0708,
        'ZCR_std': 0.0300,
        'RMSE_mean': 0.0212,
        'RMSE_std': 0.0174,
        'MFCC_means': [
            -385.598, 137.840, 1.608, 51.619, -15.655,
            22.205, -18.391, 9.706, -13.179, 3.508,
            -0.987, -4.419, 4.442, -9.774, 5.044,
            -10.782, 2.495, -8.586, 0.965, -4.649
        ],
        'MFCC_stds': [
            46.325, 14.257, 11.958, 12.195, 7.744,
            10.923, 6.812, 6.224, 4.110, 4.033,
            3.871, 3.139, 3.298, 2.954, 3.260,
            3.681, 3.200, 2.723, 2.941, 2.603
        ]
    },
    'FEAR': {
        'ZCR_mean': 0.0645,
        'ZCR_std': 0.0249,
        'RMSE_mean': 0.0303,
        'RMSE_std': 0.0310,
        'MFCC_means': [
            -379.109, 132.062, 4.086, 50.065, -14.141,
            21.857, -16.556, 8.178, -12.486, 2.971,
            -2.549, -5.420, 3.570, -9.721, 4.359,
            -10.538, 1.967, -8.359, 0.623, -4.244
        ],
        'MFCC_stds': [
            59.114, 15.984, 12.690, 12.302, 7.665,
            11.356, 6.379, 6.280, 3.946, 4.018,
            3.989, 3.531, 3.845, 3.502, 3.727,
            4.320, 3.925, 3.728, 3.632, 3.724
        ]
    },
    'HAPPY': {
        'ZCR_mean': 0.0671,
        'ZCR_std': 0.0255,
        'RMSE_mean': 0.0340,
        'RMSE_std': 0.0235,
        'MFCC_means': [
            -356.270, 133.676, 0.348, 45.186, -13.289,
            15.806, -16.239, 6.152, -13.946, 2.110,
            -2.470, -5.475, 3.416, -9.233, 4.003,
            -9.829, 1.275, -7.657, 0.534, -3.884
        ],
        'MFCC_stds': [
            47.268, 14.645, 13.520, 10.231, 8.411,
            10.349, 7.024, 5.542, 4.562, 4.244,
            4.177, 3.572, 3.609, 3.512, 3.479,
            4.405, 3.814, 3.650, 3.467, 3.484
        ]
    },
    'NEUTRAL': {
        'ZCR_mean': 0.0619,
        'ZCR_std': 0.0244,
        'RMSE_mean': 0.0167,
        'RMSE_std': 0.0062,
        'MFCC_means': [
            -398.677, 141.591, 6.849, 53.914, -14.457,
            21.870, -17.550, 8.230, -13.621, 3.048,
            -2.019, -4.408, 4.477, -9.200, 5.233,
            -10.508, 2.698, -8.454, 1.005, -4.425
        ],
        'MFCC_stds': [
            26.359, 12.097, 10.285, 8.730, 7.124,
            8.421, 6.028, 4.935, 3.831, 3.745,
            3.801, 2.985, 3.127, 2.819, 2.770,
            3.791, 2.989, 2.704, 2.824, 2.381
        ]
    },
    'SAD': {
        'ZCR_mean': 0.0553,
        'ZCR_std': 0.0218,
        'RMSE_mean': 0.0120,
        'RMSE_std': 0.0071,
        'MFCC_means': [
            -429.167, 143.558, 8.845, 58.525, -15.826,
            29.045, -17.954, 12.292, -12.491, 3.972,
            -1.606, -4.158, 4.409, -10.219, 5.771,
            -11.590, 3.432, -9.358, 1.490, -4.850
        ],
        'MFCC_stds': [
            37.168, 11.792, 9.642, 9.299, 6.888,
            9.068, 5.482, 5.340, 3.337, 3.396,
            3.323, 2.647, 2.934, 2.704, 2.720,
            3.494, 2.675, 2.329, 2.699, 2.500
        ]
    }
}


In [6]:
# import numpy as np
# import librosa
# import soundfile as sf
# from scipy.signal import savgol_filter

# emo = 'ANGRY'
# # Parameters
# sr = 22050
# duration = 2.5
# n_fft = 2048
# hop_length = 512
# n_mfcc = 20
# n_mels = 128  # Standard number of mel bands
# n_frames = 1 + (int(sr * duration) - n_fft) // hop_length  # Should be 83
# expected_samples = int(sr * duration)  # 44100 samples for 2 seconds

# # MFCC means and standard deviations
# means = emotion_data[f'{emo}']['MFCC_means']
# stds = emotion_data[f'{emo}']['MFCC_stds']
# rmse_mean = emotion_data[f'{emo}']['RMSE_mean']
# zcr_mean = emotion_data[f'{emo}']['ZCR_mean']
# zcr_std = emotion_data[f'{emo}']['ZCR_std']
# # Generate MFCC matrix with desired statistics
# M = np.zeros((n_frames, n_mfcc))
# for i in range(n_mfcc):
#     M[:, i] = means[i] + stds[i] * np.random.normal(0, 1, n_frames)
#     # Smooth over time using Savitzky-Golay filter
#     M[:, i] = savgol_filter(M[:, i], window_length=11, polyorder=2)
#     # Clip MFCC values to prevent extreme values
#     M[:, i] = np.clip(M[:, i], means[i] - 3 * stds[i], means[i] + 3 * stds[i])

# # Convert MFCC to mel-spectrogram
# mel_spec = librosa.feature.inverse.mfcc_to_mel(M, n_mels=n_mels)
# print(f"Mel-spectrogram shape: {mel_spec.shape}")  # Should be (128, 83)

# # Ensure mel-spectrogram is non-negative, finite, and stable
# mel_spec = np.maximum(mel_spec, 1e-10)  # Avoid zeros
# mel_spec = np.where(np.isfinite(mel_spec), mel_spec, 1e-10)  # Replace non-finite

# # Scale mel-spectrogram to increase energy
# mel_spec = mel_spec / (np.max(np.abs(mel_spec)) + 1e-10)  # Normalize to [0, 1]
# mel_spec = mel_spec * 1.0  # Scale for louder output

# # Reconstruct audio from mel-spectrogram
# try:
#     y = librosa.feature.inverse.mel_to_audio(
#         mel_spec, sr=sr, n_fft=n_fft, hop_length=hop_length, n_iter=100
#     )
#     print(f"Reconstructed audio shape: {y.shape}, expected: {expected_samples}")
# except Exception as e:
#     print(f"Mel-to-audio failed: {e}")
#     # Fallback: Generate a noise signal
#     y = 0.1 * np.random.normal(0, 1, expected_samples)

# # Ensure the output audio is finite and has correct length
# y = np.where(np.isfinite(y), y, 0)
# if len(y) != expected_samples:
#     print(f"Warning: Audio length {len(y)} does not match expected {expected_samples}")
#     y = np.pad(y, (0, max(0, expected_samples - len(y))), mode='constant')[:expected_samples]

# # Add a small noise component to ensure audibility
# t = np.linspace(0, duration, len(y))
# noise = 0.05 * np.random.normal(0.0846, 0.0296, len(y))  # Increased noise for ZCR
# y += noise
# y = np.where(np.isfinite(y), y, 0)

# # Adjust amplitude to match RMSE mean
# current_rmse = np.sqrt(np.mean(y**2))
# target_rmse_mean = 0.0692
# scaling_factor = target_rmse_mean / (current_rmse + 1e-10)
# y_scaled = y * scaling_factor

# # # Normalize to typical audio range (-1 to 1) for audibility
# # y_scaled = y_scaled / (np.max(np.abs(y_scaled)) + 1e-10) * 0.5  # Scale to ±0.5
# # y_scaled = np.where(np.isfinite(y_scaled), y_scaled, 0)

# # Save the audio
# output_path = "synthetic_audio.wav"  # Adjust path as needed
# sf.write(output_path, y_scaled, sr)
# print(f"Audio saved to: {output_path}")

# # Verify file duration
# info = sf.info(output_path)
# print(f"File duration: {info.duration:.2f} seconds, expected: {duration:.2f} seconds")

# # Verify statistics
# zcr = librosa.feature.zero_crossing_rate(y_scaled, hop_length=hop_length)
# rmse = librosa.feature.rms(y=y_scaled, hop_length=hop_length)
# mfccs = librosa.feature.mfcc(y=y_scaled, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length)

# print(f"ZCR Mean: {zcr.mean():.4f}, Std: {zcr.std():.4f}")
# print(f"RMSE Mean: {rmse.mean():.4f}, Std: {rmse.std():.4f}")
# for i in range(n_mfcc):
#     print(f"MFCC {i+1} Mean: {mfccs[i].mean():.3f}, Std: {mfccs[i].std():.3f}")

# # Check signal amplitude
# print(f"Max amplitude: {np.max(np.abs(y_scaled)):.6f}")
# print(f"Mean amplitude: {np.mean(np.abs(y_scaled)):.6f}")

In [7]:
# import numpy as np
# import librosa
# import soundfile as sf
# from scipy.signal import savgol_filter

# emo = 'HAPPY'

# # Parameters
# sr = 22050
# duration = 2.5
# n_fft = 2048
# hop_length = 512
# n_mfcc = 20
# n_mels = 128  # Standard number of mel bands
# n_frames = 1 + (int(sr * duration) - n_fft) // hop_length  # Should be ~103
# expected_samples = int(sr * duration)  # 55125 samples for 2.5 seconds

# # Target statistics
# means = emotion_data[f'{emo}']['MFCC_means']
# stds = emotion_data[f'{emo}']['MFCC_stds']
# rmse_mean = emotion_data[f'{emo}']['RMSE_mean']
# rmse_std = emotion_data[f'{emo}']['RMSE_std']
# zcr_mean = emotion_data[f'{emo}']['ZCR_mean']
# zcr_std = emotion_data[f'{emo}']['ZCR_std']

# # Generate MFCC matrix with desired statistics
# M = np.zeros((n_frames, n_mfcc))
# for i in range(n_mfcc):
#     M[:, i] = means[i] + stds[i] * np.random.normal(0, 1, n_frames)
#     # Smooth over time using Savitzky-Golay filter
#     M[:, i] = savgol_filter(M[:, i], window_length=11, polyorder=2)
#     # Clip MFCC values to prevent extreme values
#     M[:, i] = np.clip(M[:, i], means[i] - 3 * stds[i], means[i] + 3 * stds[i])

# # Convert MFCC to mel-spectrogram
# mel_spec = librosa.feature.inverse.mfcc_to_mel(M, n_mels=n_mels)
# print(f"Mel-spectrogram shape: {mel_spec.shape}")  # Should be (128, ~103)

# # Ensure mel-spectrogram is non-negative, finite, and stable
# mel_spec = np.maximum(mel_spec, 1e-10)  # Avoid zeros
# mel_spec = np.where(np.isfinite(mel_spec), mel_spec, 1e-10)  # Replace non-finite

# # Scale mel-spectrogram to increase energy
# mel_spec = mel_spec / (np.max(np.abs(mel_spec)) + 1e-10)  # Normalize to [0, 1]
# mel_spec = mel_spec * 1.0  # Scale for louder output

# # Reconstruct audio from mel-spectrogram
# try:
#     y = librosa.feature.inverse.mel_to_audio(
#         mel_spec, sr=sr, n_fft=n_fft, hop_length=hop_length, n_iter=100
#     )
#     print(f"Reconstructed audio shape: {y.shape}, expected: {expected_samples}")
# except Exception as e:
#     print(f"Mel-to-audio failed: {e}")
#     # Fallback: Generate a noise signal
#     y = 0.1 * np.random.normal(0, 1, expected_samples)

# # Ensure the output audio is finite and has correct length
# y = np.where(np.isfinite(y), y, 0)
# if len(y) != expected_samples:
#     print(f"Warning: Audio length {len(y)} does not match expected {expected_samples}")
#     y = np.pad(y, (0, max(0, expected_samples - len(y))), mode='constant')[:expected_samples]

# # Add noise to adjust ZCR
# t = np.linspace(0, duration, len(y))
# noise = 0.05 * np.random.normal(0, 1, len(y))  # Noise to increase ZCR
# y += noise
# y = np.where(np.isfinite(y), y, 0)

# # Apply envelope to match RMSE std
# # Generate a smooth envelope to vary amplitude across frames
# envelope = np.ones(len(y))
# frame_indices = np.arange(0, len(y), hop_length)
# envelope_frames = np.random.normal(rmse_mean, rmse_std, len(frame_indices))
# envelope_frames = savgol_filter(envelope_frames, window_length=11, polyorder=2)  # Smooth envelope
# envelope_frames = np.maximum(envelope_frames, 1e-10)  # Ensure non-negative
# for i in range(len(frame_indices) - 1):
#     start = frame_indices[i]
#     end = frame_indices[i + 1]
#     envelope[start:end] = np.linspace(envelope_frames[i], envelope_frames[i + 1], end - start)
# # Interpolate last frame
# envelope[frame_indices[-1]:] = envelope_frames[-1]
# y = y * envelope
# y = np.where(np.isfinite(y), y, 0)

# # Adjust amplitude to match RMSE mean
# current_rmse = np.sqrt(np.mean(y**2))
# scaling_factor = rmse_mean / (current_rmse + 1e-10)
# y_scaled = y * scaling_factor
# y_scaled = np.where(np.isfinite(y_scaled), y_scaled, 0)

# # Normalize to typical audio range (-1 to 1) for audibility
# max_amplitude = np.max(np.abs(y_scaled))
# if max_amplitude > 0:
#     y_scaled = y_scaled / max_amplitude * 0.5  # Scale to ±0.5
# y_scaled = np.where(np.isfinite(y_scaled), y_scaled, 0)

# # Save the audio
# output_path = "synthetic_audio.wav"  # Adjust path as needed
# sf.write(output_path, y_scaled, sr)
# print(f"Audio saved to: {output_path}")

# # Verify file duration
# info = sf.info(output_path)
# print(f"File duration: {info.duration:.2f} seconds, expected: {duration:.2f} seconds")

# # Verify statistics
# zcr = librosa.feature.zero_crossing_rate(y_scaled, hop_length=hop_length)
# rmse = librosa.feature.rms(y=y_scaled, hop_length=hop_length)
# mfccs = librosa.feature.mfcc(y=y_scaled, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length)

# print(f"ZCR Mean: {zcr.mean():.4f}, Std: {zcr.std():.4f}")
# print(f"RMSE Mean: {rmse.mean():.4f}, Std: {rmse.std():.4f}")
# for i in range(n_mfcc):
#     print(f"MFCC {i+1} Mean: {mfccs[i].mean():.3f}, Std: {mfccs[i].std():.3f}")

# # Check signal amplitude
# print(f"Max amplitude: {np.max(np.abs(y_scaled)):.6f}")
# print(f"Mean amplitude: {np.mean(np.abs(y_scaled)):.6f}")

In [8]:
# import numpy as np
# import librosa
# import soundfile as sf
# from scipy.signal import savgol_filter

# # User input: specify the emotion (ANGRY, DISGUST, FEAR, HAPPY, NEUTRAL, SAD)
# emo = 'DISGUST'  # Change this to the desired emotion

# # Parameters
# sr = 22050
# duration = 2.5
# n_fft = 2048
# hop_length = 512
# n_mfcc = 20
# n_mels = 128
# expected_samples = int(sr * duration)
# n_frames = 1 + (expected_samples - n_fft) // hop_length

# # Dictionary containing audio features for different emotions
# # emotion_data should be defined elsewhere with all emotion statistics

# # Target statistics
# means = emotion_data[f'{emo}']['MFCC_means']
# stds = emotion_data[f'{emo}']['MFCC_stds']
# target_rmse_mean = emotion_data[f'{emo}']['RMSE_mean']
# target_rmse_std = emotion_data[f'{emo}']['RMSE_std']
# target_zcr_mean = emotion_data[f'{emo}']['ZCR_mean']
# target_zcr_std = emotion_data[f'{emo}']['ZCR_std']

# print(f"Generating {emo} audio with target statistics:")
# print(f"ZCR Mean: {target_zcr_mean:.4f}, Std: {target_zcr_std:.4f}")
# print(f"RMSE Mean: {target_rmse_mean:.4f}, Std: {target_rmse_std:.4f}")

# # Function to evaluate how close the generated audio matches target statistics
# def evaluate_match(y, target_zcr_mean, target_zcr_std, target_rmse_mean, target_rmse_std, means, stds):
#     zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop_length).flatten()
#     rmse = librosa.feature.rms(y=y, hop_length=hop_length).flatten()
#     mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length)
    
#     zcr_diff = abs(zcr.mean() - target_zcr_mean) + abs(zcr.std() - target_zcr_std)
#     rmse_diff = abs(rmse.mean() - target_rmse_mean) + abs(rmse.std() - target_rmse_std)
    
#     mfcc_diff = 0
#     for i in range(n_mfcc):
#         mfcc_diff += abs(mfccs[i].mean() - means[i]) / (abs(means[i]) + 1e-10)
#         mfcc_diff += abs(mfccs[i].std() - stds[i]) / (stds[i] + 1e-10)
    
#     return zcr_diff + rmse_diff + mfcc_diff/40, zcr.mean(), zcr.std(), rmse.mean(), rmse.std()

# # Generate multiple candidates and select the best one
# best_score = float('inf')
# best_audio = None
# best_stats = None
# n_candidates = 10

# for candidate in range(n_candidates):
#     print(f"Generating candidate {candidate+1}/{n_candidates}...")
    
#     # 1. Initial MFCC generation with target statistics
#     M = np.zeros((n_mfcc, n_frames))
#     for i in range(n_mfcc):
#         M[i, :] = means[i] + stds[i] * np.random.normal(0, 1, n_frames)
#         # Apply smoothing for temporal coherence
#         M[i, :] = savgol_filter(M[i, :], window_length=min(11, n_frames-1), polyorder=min(2, n_frames-2))
#         # Clip to prevent extreme values
#         M[i, :] = np.clip(M[i, :], means[i] - 3 * stds[i], means[i] + 3 * stds[i])
    
#     # 2. Convert MFCC to mel-spectrogram
#     # Use more reliable inversion - directly to audio
#     y = librosa.feature.inverse.mfcc_to_audio(
#         M, sr=sr, n_fft=n_fft, hop_length=hop_length, 
#         n_iter=50, norm=None
#     )
    
#     # Ensure correct length
#     if len(y) < expected_samples:
#         y = np.pad(y, (0, expected_samples - len(y)), mode='constant')
#     else:
#         y = y[:expected_samples]
    
#     # 3. Add controlled noise to adjust ZCR
#     # First measure current ZCR
#     current_zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop_length).mean()
    
#     # Adjust noise level based on difference between current and target ZCR
#     if current_zcr < target_zcr_mean:
#         # Add high-frequency noise to increase ZCR
#         noise_strength = min(0.2, abs(target_zcr_mean - current_zcr) * 2)
#         high_freq_noise = noise_strength * np.random.normal(0, 1, len(y))
#         y = y + high_freq_noise
    
#     # 4. Apply dynamic envelope to match RMSE statistics
#     # Generate envelope frames with target RMSE statistics
#     frame_indices = np.arange(0, len(y), hop_length)
#     envelope_frames = np.abs(np.random.normal(target_rmse_mean, target_rmse_std, len(frame_indices)))
#     envelope_frames = savgol_filter(envelope_frames, 
#                                     window_length=min(11, len(envelope_frames)-1), 
#                                     polyorder=min(2, len(envelope_frames)-2))
    
#     # Interpolate envelope across all samples
#     envelope = np.interp(np.arange(len(y)), frame_indices, envelope_frames)
    
#     # Apply envelope
#     y = y * envelope
    
#     # 5. Normalize for typical audio range
#     # First measure current RMSE
#     current_rmse = np.sqrt(np.mean(y**2))
    
#     # Scale to match target RMSE mean
#     scaling_factor = target_rmse_mean / (current_rmse + 1e-10)
#     y_scaled = y * scaling_factor
    
#     # Final normalization for audio quality
#     max_amp = np.max(np.abs(y_scaled))
#     if max_amp > 0.95:
#         y_scaled = y_scaled / max_amp * 0.95
    
#     # Clean up any NaN or inf values
#     y_scaled = np.where(np.isfinite(y_scaled), y_scaled, 0)
    
#     # Evaluate how well this candidate matches target statistics
#     score, curr_zcr_mean, curr_zcr_std, curr_rmse_mean, curr_rmse_std = evaluate_match(
#         y_scaled, target_zcr_mean, target_zcr_std, target_rmse_mean, target_rmse_std, means, stds
#     )
    
#     print(f"  Score: {score:.4f}")
#     print(f"  ZCR: {curr_zcr_mean:.4f} (target: {target_zcr_mean:.4f})")
#     print(f"  RMSE: {curr_rmse_mean:.4f} (target: {target_rmse_mean:.4f})")
    
#     if score < best_score:
#         best_score = score
#         best_audio = y_scaled
#         best_stats = (curr_zcr_mean, curr_zcr_std, curr_rmse_mean, curr_rmse_std)

# # Use the best candidate
# y_final = best_audio
# print(f"\nBest candidate score: {best_score:.4f}")
# print(f"ZCR Mean: {best_stats[0]:.4f} (target: {target_zcr_mean:.4f}), Std: {best_stats[1]:.4f} (target: {target_zcr_std:.4f})")
# print(f"RMSE Mean: {best_stats[2]:.4f} (target: {target_rmse_mean:.4f}), Std: {best_stats[3]:.4f} (target: {target_rmse_std:.4f})")

# # Save the audio
# output_path = f"synthetic_{emo}_audio.wav"
# sf.write(output_path, y_final, sr)
# print(f"Audio saved to: {output_path}")

# # Verify file statistics
# print("\nFinal audio statistics:")
# zcr = librosa.feature.zero_crossing_rate(y_final, hop_length=hop_length)
# rmse = librosa.feature.rms(y=y_final, hop_length=hop_length)
# mfccs = librosa.feature.mfcc(y=y_final, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length)

# print(f"ZCR Mean: {zcr.mean():.4f}, Std: {zcr.std():.4f}")
# print(f"RMSE Mean: {rmse.mean():.4f}, Std: {rmse.std():.4f}")
# for i in range(n_mfcc):
#     print(f"MFCC {i+1} Mean: {mfccs[i].mean():.3f}, Std: {mfccs[i].std():.3f}, Target: {means[i]:.3f} ± {stds[i]:.3f}")

# # Check signal properties
# print(f"File duration: {len(y_final)/sr:.2f} seconds, expected: {duration:.2f} seconds")
# print(f"Max amplitude: {np.max(np.abs(y_final)):.6f}")
# print(f"Mean amplitude: {np.mean(np.abs(y_final)):.6f}")

In [9]:
# import numpy as np
# import librosa
# import soundfile as sf
# from scipy.signal import savgol_filter

# # User input: specify the emotion (ANGRY, DISGUST, FEAR, HAPPY, NEUTRAL, SAD)
# emo = 'ANGRY'  # Change this to the desired emotion

# # Parameters
# sr = 22050
# duration = 2.5
# n_fft = 2048
# hop_length = 512
# n_mfcc = 20
# n_mels = 128
# expected_samples = int(sr * duration)
# n_frames = 1 + (expected_samples - n_fft) // hop_length

# # Dictionary containing audio features for different emotions
# # emotion_data should be defined elsewhere with all emotion statistics

# # Target statistics
# means = emotion_data[f'{emo}']['MFCC_means']
# stds = emotion_data[f'{emo}']['MFCC_stds']
# target_rmse_mean = emotion_data[f'{emo}']['RMSE_mean']
# target_rmse_std = emotion_data[f'{emo}']['RMSE_std']
# target_zcr_mean = emotion_data[f'{emo}']['ZCR_mean']
# target_zcr_std = emotion_data[f'{emo}']['ZCR_std']

# print(f"Generating {emo} audio with target statistics:")
# print(f"ZCR Mean: {target_zcr_mean:.4f}, Std: {target_zcr_std:.4f}")
# print(f"RMSE Mean: {target_rmse_mean:.4f}, Std: {target_rmse_std:.4f}")

# # Function to evaluate how close the generated audio matches target statistics
# def evaluate_match(y, target_zcr_mean, target_zcr_std, target_rmse_mean, target_rmse_std, means, stds):
#     zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop_length).flatten()
#     rmse = librosa.feature.rms(y=y, hop_length=hop_length).flatten()
#     mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length)
    
#     # Calculate relative differences for ZCR and RMSE
#     zcr_diff = abs(zcr.mean() - target_zcr_mean) / (target_zcr_mean + 1e-10)
#     zcr_std_diff = abs(zcr.std() - target_zcr_std) / (target_zcr_std + 1e-10)
#     rmse_diff = abs(rmse.mean() - target_rmse_mean) / (target_rmse_mean + 1e-10)
#     rmse_std_diff = abs(rmse.std() - target_rmse_std) / (target_rmse_std + 1e-10)
    
#     # Calculate weighted MFCC differences (especially prioritize lower coefficients)
#     mfcc_diff = 0
#     weights = np.linspace(1, 1.0, n_mfcc)  # Higher weights for lower coefficients
#     for i in range(n_mfcc):
#         # Normalized mean difference
#         mean_diff = abs(mfccs[i].mean() - means[i]) / (abs(means[i]) + 1e-10)
#         # Normalized std difference
#         std_diff = abs(mfccs[i].std() - stds[i]) / (stds[i] + 1e-10)
#         # Apply weight
#         mfcc_diff += weights[i] * (mean_diff + std_diff)
    
#     # Combine scores with higher weight on MFCCs
#     zcr_rmse_score = zcr_diff + zcr_std_diff + rmse_diff + rmse_std_diff
#     mfcc_score = mfcc_diff / n_mfcc  # Normalize by number of coefficients
    
#     # Final weighted score - MFCC has 3x the importance of ZCR and RMSE combined
#     final_score = (zcr_rmse_score * 1.0) + (mfcc_score * 3.0)
    
#     return final_score, zcr.mean(), zcr.std(), rmse.mean(), rmse.std()

# # Generate multiple candidates and select the best one
# best_score = float('inf')
# best_audio = None
# best_stats = None
# best_mfccs = None
# n_candidates = 100

# for candidate in range(n_candidates):
#     print(f"Generating candidate {candidate+1}/{n_candidates}...")
    
#     # 1. Initial MFCC generation with target statistics
#     M = np.zeros((n_mfcc, n_frames))
#     for i in range(n_mfcc):
#         M[i, :] = means[i] + stds[i] * np.random.normal(0, 1, n_frames)
#         # Apply smoothing for temporal coherence
#         M[i, :] = savgol_filter(M[i, :], window_length=min(11, n_frames-1), polyorder=min(2, n_frames-2))
#         # Clip to prevent extreme values
#         M[i, :] = np.clip(M[i, :], means[i] - 3 * stds[i], means[i] + 3 * stds[i])
    
#     # 2. Convert MFCC to mel-spectrogram
#     # Use more reliable inversion - directly to audio
#     y = librosa.feature.inverse.mfcc_to_audio(
#         M, sr=sr, n_fft=n_fft, hop_length=hop_length, 
#         n_iter=50, norm=None
#     )
    
#     # Ensure correct length
#     if len(y) < expected_samples:
#         y = np.pad(y, (0, expected_samples - len(y)), mode='constant')
#     else:
#         y = y[:expected_samples]
    
#     # 3. Add controlled noise to adjust ZCR
#     # First measure current ZCR
#     current_zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop_length).mean()
    
#     # Adjust noise level based on difference between current and target ZCR
#     if current_zcr < target_zcr_mean:
#         # Add high-frequency noise to increase ZCR
#         noise_strength = min(0.2, abs(target_zcr_mean - current_zcr) * 2)
#         high_freq_noise = noise_strength * np.random.normal(0, 1, len(y))
#         y = y + high_freq_noise
    
#     # 4. Apply dynamic envelope to match RMSE statistics
#     # Generate envelope frames with target RMSE statistics
#     frame_indices = np.arange(0, len(y), hop_length)
#     envelope_frames = np.abs(np.random.normal(target_rmse_mean, target_rmse_std, len(frame_indices)))
#     envelope_frames = savgol_filter(envelope_frames, 
#                                     window_length=min(11, len(envelope_frames)-1), 
#                                     polyorder=min(2, len(envelope_frames)-2))
    
#     # Interpolate envelope across all samples
#     envelope = np.interp(np.arange(len(y)), frame_indices, envelope_frames)
    
#     # Apply envelope
#     y = y * envelope
    
#     # 5. Normalize for typical audio range
#     # First measure current RMSE
#     current_rmse = np.sqrt(np.mean(y**2))
    
#     # Scale to match target RMSE mean
#     scaling_factor = target_rmse_mean / (current_rmse + 1e-10)
#     y_scaled = y * scaling_factor
    
#     # Additional MFCC-based tuning iterations - repeatedly check and adjust
#     for tuning_iter in range(2):
#         # Calculate current MFCCs
#         current_mfccs = librosa.feature.mfcc(y=y_scaled, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length)
        
#         # Create a correction factor for each MFCC
#         correction = np.ones((n_mfcc, n_frames))
#         for i in range(n_mfcc):
#             # Calculate how much we need to adjust each MFCC
#             if abs(current_mfccs[i].mean()) > 1e-10:
#                 mean_ratio = means[i] / (current_mfccs[i].mean() + 1e-10)
#                 # Apply partial correction (don't adjust too drastically)
#                 frame_correction = np.ones(n_frames) * (0.3 * mean_ratio + 0.7)
#                 correction[i, :] = frame_correction
        
#         # Convert correction factors to audio adjustment
#         if tuning_iter == 0:
#             # First iteration more aggressive
#             correction_strength = 0.3
#         else:
#             # Second iteration more subtle
#             correction_strength = 0.15
            
#         # Apply gentle equalization based on MFCC discrepancies
#         y_equalized = y_scaled.copy()
#         # Create band-specific gain adjustments
#         for i in range(n_mfcc):
#             # Create band-pass filtered version of signal to adjust
#             band_center_hz = 500 * (i + 1)  # Rough mapping of MFCC index to frequency
#             band_width = band_center_hz * 0.5
            
#             # Don't adjust DC component (first MFCC) directly
#             if i > 0:
#                 # Simple band adjustment using bandpass filter
#                 y_band = librosa.effects.preemphasis(y_scaled, coef=0.95 if i > 3 else 0.5)
#                 # Gentle adjustment based on mean difference
#                 mean_diff = means[i] - current_mfccs[i].mean()
#                 # Scale adjustment by importance of coefficient
#                 importance = max(0.1, 1.0 - (i/n_mfcc))
#                 adjustment = mean_diff * importance * correction_strength
#                 # Apply adjustment
#                 y_equalized = y_equalized + adjustment * y_band
        
#         # Re-normalize
#         max_amp = np.max(np.abs(y_equalized))
#         if max_amp > 0.95:
#             y_equalized = y_equalized / max_amp * 0.95
            
#         # Update scaled audio
#         y_scaled = y_equalized
    
#     # Final normalization for audio quality
#     max_amp = np.max(np.abs(y_scaled))
#     if max_amp > 0.95:
#         y_scaled = y_scaled / max_amp * 0.95
    
#     # Clean up any NaN or inf values
#     y_scaled = np.where(np.isfinite(y_scaled), y_scaled, 0)
    
#     # Evaluate how well this candidate matches target statistics
#     score, curr_zcr_mean, curr_zcr_std, curr_rmse_mean, curr_rmse_std = evaluate_match(
#         y_scaled, target_zcr_mean, target_zcr_std, target_rmse_mean, target_rmse_std, means, stds
#     )
    
#     # Store current MFCCs for detailed reporting if this is the best candidate
#     current_mfccs = librosa.feature.mfcc(y=y_scaled, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length)
    
#     print(f"  Score: {score:.4f}")
#     print(f"  ZCR: {curr_zcr_mean:.4f} (target: {target_zcr_mean:.4f})")
#     print(f"  RMSE: {curr_rmse_mean:.4f} (target: {target_rmse_mean:.4f})")
    
#     if score < best_score:
#         best_score = score
#         best_audio = y_scaled
#         best_stats = (curr_zcr_mean, curr_zcr_std, curr_rmse_mean, curr_rmse_std)
#         best_mfccs = current_mfccs

# # Use the best candidate
# y_final = best_audio
# print(f"\nBest candidate score: {best_score:.4f}")
# print(f"ZCR Mean: {best_stats[0]:.4f} (target: {target_zcr_mean:.4f}), Std: {best_stats[1]:.4f} (target: {target_zcr_std:.4f})")
# print(f"RMSE Mean: {best_stats[2]:.4f} (target: {target_rmse_mean:.4f}), Std: {best_stats[3]:.4f} (target: {target_rmse_std:.4f})")

# # Print MFCC comparison table for best candidate
# print("\nMFCC Statistics Comparison:")
# print(f"{'MFCC #':<7} {'Current Mean':>12} {'Target Mean':>12} {'Diff %':>8} {'Current Std':>12} {'Target Std':>12} {'Diff %':>8}")
# print("-" * 70)
# for i in range(n_mfcc):
#     curr_mean = best_mfccs[i].mean()
#     curr_std = best_mfccs[i].std()
#     target_mean = means[i]
#     target_std = stds[i]
    
#     mean_diff_pct = abs(curr_mean - target_mean) / (abs(target_mean) + 1e-10) * 100
#     std_diff_pct = abs(curr_std - target_std) / (target_std + 1e-10) * 100
    
#     print(f"{i+1:<7} {curr_mean:>12.3f} {target_mean:>12.3f} {mean_diff_pct:>7.1f}% {curr_std:>12.3f} {target_std:>12.3f} {std_diff_pct:>7.1f}%")

# # Save the audio
# output_path = f"synthetic_{emo}_audio.wav"
# sf.write(output_path, y_final, sr)
# print(f"\nAudio saved to: {output_path}")

# # Verify file statistics
# print("\nFinal audio statistics:")
# zcr = librosa.feature.zero_crossing_rate(y_final, hop_length=hop_length)
# rmse = librosa.feature.rms(y=y_final, hop_length=hop_length)

# print(f"ZCR Mean: {zcr.mean():.4f}, Std: {zcr.std():.4f}")
# print(f"RMSE Mean: {rmse.mean():.4f}, Std: {rmse.std():.4f}")

# # Check signal properties
# print(f"File duration: {len(y_final)/sr:.2f} seconds, expected: {duration:.2f} seconds")
# print(f"Max amplitude: {np.max(np.abs(y_final)):.6f}")
# print(f"Mean amplitude: {np.mean(np.abs(y_final)):.6f}")

In [10]:
# import numpy as np
# import librosa
# import soundfile as sf
# import pickle
# import tensorflow as tf
# from scipy import signal

# # Define the dictionary structure based on the provided data
# def create_emotion_data_dict():
#     data = {}
    
#     # ZCR data
#     zcr_means = {
#         'ANGRY': 0.0846, 'DISGUST': 0.0708, 'FEAR': 0.0645,
#         'HAPPY': 0.0671, 'NEUTRAL': 0.0619, 'SAD': 0.0553
#     }
    
#     zcr_stds = {
#         'ANGRY': 0.0296, 'DISGUST': 0.0300, 'FEAR': 0.0249,
#         'HAPPY': 0.0255, 'NEUTRAL': 0.0244, 'SAD': 0.0218
#     }
    
#     # RMSE data
#     rmse_means = {
#         'ANGRY': 0.0692, 'DISGUST': 0.0212, 'FEAR': 0.0303,
#         'HAPPY': 0.0340, 'NEUTRAL': 0.0167, 'SAD': 0.0120
#     }
    
#     rmse_stds = {
#         'ANGRY': 0.0478, 'DISGUST': 0.0174, 'FEAR': 0.0310,
#         'HAPPY': 0.0235, 'NEUTRAL': 0.0062, 'SAD': 0.0071
#     }
    
#     # MFCC data (organized by emotion)
#     mfcc_means = {
#         'ANGRY': [
#             -298.124, 121.593, -7.995, 41.206, -17.941, 9.999, -15.744, 4.588, -15.002, 1.481,
#             -3.137, -5.640, 3.305, -9.326, 2.893, -9.847, 0.812, -8.127, 0.116, -3.868
#         ],
#         'DISGUST': [
#             -385.598, 137.840, 1.608, 51.619, -15.655, 22.205, -18.391, 9.706, -13.179, 3.508,
#             -0.987, -4.419, 4.442, -9.774, 5.044, -10.782, 2.495, -8.586, 0.965, -4.649
#         ],
#         'FEAR': [
#             -379.109, 132.062, 4.086, 50.065, -14.141, 21.857, -16.556, 8.178, -12.486, 2.971,
#             -2.549, -5.420, 3.570, -9.721, 4.359, -10.538, 1.967, -8.359, 0.623, -4.244
#         ],
#         'HAPPY': [
#             -356.270, 133.676, 0.348, 45.186, -13.289, 15.806, -16.239, 6.152, -13.946, 2.110,
#             -2.470, -5.475, 3.416, -9.233, 4.003, -9.829, 1.275, -7.657, 0.534, -3.884
#         ],
#         'NEUTRAL': [
#             -398.677, 141.591, 6.849, 53.914, -14.457, 21.870, -17.550, 8.230, -13.621, 3.048,
#             -2.019, -4.408, 4.477, -9.200, 5.233, -10.508, 2.698, -8.454, 1.005, -4.425
#         ],
#         'SAD': [
#             -429.167, 143.558, 8.845, 58.525, -15.826, 29.045, -17.954, 12.292, -12.491, 3.972,
#             -1.606, -4.158, 4.409, -10.219, 5.771, -11.590, 3.432, -9.358, 1.490, -4.850
#         ]
#     }
    
#     mfcc_stds = {
#         'ANGRY': [
#             57.716, 17.098, 15.712, 11.359, 9.764, 10.442, 7.793, 5.566, 4.787, 4.461,
#             4.062, 3.706, 3.975, 3.546, 3.853, 4.285, 4.058, 3.810, 3.839, 3.803
#         ],
#         'DISGUST': [
#             46.325, 14.257, 11.958, 12.195, 7.744, 10.923, 6.812, 6.224, 4.110, 4.033,
#             3.871, 3.139, 3.298, 2.954, 3.260, 3.681, 3.200, 2.723, 2.941, 2.603
#         ],
#         'FEAR': [
#             59.114, 15.984, 12.690, 12.302, 7.665, 11.356, 6.379, 6.280, 3.946, 4.018,
#             3.989, 3.531, 3.845, 3.502, 3.727, 4.320, 3.925, 3.728, 3.632, 3.724
#         ],
#         'HAPPY': [
#             47.268, 14.645, 13.520, 10.231, 8.411, 10.349, 7.024, 5.542, 4.562, 4.244,
#             4.177, 3.572, 3.609, 3.512, 3.479, 4.405, 3.814, 3.650, 3.467, 3.484
#         ],
#         'NEUTRAL': [
#             26.359, 12.097, 10.285, 8.730, 7.124, 8.421, 6.028, 4.935, 3.831, 3.745,
#             3.801, 2.985, 3.127, 2.819, 2.770, 3.791, 2.989, 2.704, 2.824, 2.381
#         ],
#         'SAD': [
#             37.168, 11.792, 9.642, 9.299, 6.888, 9.068, 5.482, 5.340, 3.337, 3.396,
#             3.323, 2.647, 2.934, 2.704, 2.720, 3.494, 2.675, 2.329, 2.699, 2.500
#         ]
#     }
    
#     for emotion in zcr_means.keys():
#         data[emotion] = {
#             'ZCR_mean': zcr_means[emotion],
#             'ZCR_std': zcr_stds[emotion],
#             'RMSE_mean': rmse_means[emotion],
#             'RMSE_std': rmse_stds[emotion],
#             'MFCC_means': mfcc_means[emotion],
#             'MFCC_stds': mfcc_stds[emotion]
#         }
    
#     return data

# def generate_synthetic_audio(emotion, duration=2.5, sr=22050, data=None):
#     """
#     Generate synthetic audio with characteristics matching a specific emotion
#     """
#     if data is None:
#         data = create_emotion_data_dict()
    
#     # Get the emotion data
#     emotion_data = data[emotion]
    
#     # Create a base audio signal
#     t = np.arange(0, duration, 1/sr)
    
#     # We'll use a mix of frequencies to create a synthetic speech-like signal
#     # Start with vocal tract resonance frequencies (formants) for basic speech-like sound
#     f1 = 500  # First formant
#     f2 = 1500  # Second formant
#     f3 = 2500  # Third formant
    
#     # Create a base signal with formants
#     audio = 0.1 * np.sin(2 * np.pi * f1 * t)
#     audio += 0.05 * np.sin(2 * np.pi * f2 * t)
#     audio += 0.025 * np.sin(2 * np.pi * f3 * t)
    
#     # Add some random fluctuations to simulate voice variations
#     fluctuation = np.random.normal(0, 0.01, len(audio))
#     audio += fluctuation
    
#     # Apply amplitude modulation based on RMSE
#     rmse_mean = emotion_data['RMSE_mean']
#     rmse_variation = np.random.normal(rmse_mean, emotion_data['RMSE_std'], len(t))
#     rmse_variation = np.abs(rmse_variation)  # Ensure positive values
    
#     # Create an envelope using the RMSE pattern
#     envelope = np.interp(np.linspace(0, 1, len(audio)), np.linspace(0, 1, 100), 
#                           np.random.normal(rmse_mean, emotion_data['RMSE_std'], 100))
#     envelope = np.abs(envelope)
    
#     # Apply the envelope
#     audio *= envelope[:len(audio)]
    
#     # Adjust overall amplitude
#     if emotion == 'ANGRY':
#         # Make angry louder
#         audio *= 1.5
#     elif emotion == 'SAD':
#         # Make sad quieter
#         audio *= 0.7
    
#     # Add some noise with ZCR characteristics
#     zcr_factor = emotion_data['ZCR_mean'] * 10  # Scale ZCR for noise addition
#     noise = np.random.normal(0, zcr_factor, len(audio))
#     audio += noise
    
#     # Apply some filtering based on MFCCs
#     # We'll use a simple filter bank approach
#     mfcc_means = emotion_data['MFCC_means']
    
#     # Create different frequency components based on the first few MFCCs
#     for i in range(min(5, len(mfcc_means))):
#         freq = 300 + i * 200  # Simple frequency spacing
#         amp = np.abs(mfcc_means[i] / 500)  # Scale MFCC to reasonable amplitude
#         audio += amp * np.sin(2 * np.pi * freq * t)
    
#     # Apply vocal characteristics based on emotion
#     if emotion == 'ANGRY':
#         # Add more high frequency content and roughness
#         roughness = np.random.normal(0, 0.2, len(audio))
#         audio += roughness
#         # Add trembling effect for anger
#         tremolo = 1 + 0.2 * np.sin(2 * np.pi * 12 * t)
#         audio *= tremolo
        
#     elif emotion == 'SAD':
#         # Add slower modulation for sadness
#         slow_mod = 1 + 0.1 * np.sin(2 * np.pi * 3 * t)
#         audio *= slow_mod
#         # Filter to reduce high frequencies
#         b, a = signal.butter(3, 0.3, 'low')
#         audio = signal.lfilter(b, a, audio)
        
#     elif emotion == 'HAPPY':
#         # Add faster modulation for happiness
#         fast_mod = 1 + 0.15 * np.sin(2 * np.pi * 8 * t)
#         audio *= fast_mod
#         # Boost mid frequencies
#         b, a = signal.butter(3, [0.1, 0.7], 'band')
#         audio = signal.lfilter(b, a, audio)
        
#     elif emotion == 'FEAR':
#         # Add irregular trembling
#         irreg_trem = 1 + 0.25 * np.sin(2 * np.pi * (5 + np.sin(2 * np.pi * 0.5 * t)) * t)
#         audio *= irreg_trem
#         # Add some breathiness
#         breathiness = np.random.normal(0, 0.15, len(audio))
#         audio += breathiness
        
#     elif emotion == 'DISGUST':
#         # Add roughness
#         roughness = np.random.normal(0, 0.15, len(audio))
#         audio += roughness
#         # Add some distortion
#         audio = np.tanh(1.5 * audio)
        
#     # Normalize
#     audio = audio / np.max(np.abs(audio))
    
#     return audio

# def create_all_emotion_audio(output_dir="emotion_audio", sr=22050):
#     """Create audio files for all emotions"""
#     import os
    
#     if not os.path.exists(output_dir):
#         os.makedirs(output_dir)
    
#     data = create_emotion_data_dict()
#     emotions = list(data.keys())
    
#     for emotion in emotions:
#         print(f"Generating {emotion} audio...")
#         audio = generate_synthetic_audio(emotion, duration=2.5, sr=sr, data=data)
#         output_file = os.path.join(output_dir, f"{emotion.lower()}_synthetic.wav")
#         sf.write(output_file, audio, sr)
#         print(f"Saved to {output_file}")

# def extract_features(data, sr=22050, frame_length=2048, hop_length=512):
#     """
#     Extract features (ZCR, RMSE, MFCC) from the audio data
#     """
#     # Zero Crossing Rate
#     zcr = librosa.feature.zero_crossing_rate(data, frame_length=frame_length, hop_length=hop_length)
    
#     # Root Mean Square Energy
#     rmse = librosa.feature.rms(y=data, frame_length=frame_length, hop_length=hop_length)
    
#     # MFCCs
#     mfcc_features = librosa.feature.mfcc(y=data, sr=sr, n_mfcc=20, n_fft=frame_length, hop_length=hop_length)
#     mfcc_features = mfcc_features.T  # Transpose to get time x features
    
#     # Flatten all features for easier processing
#     result = np.hstack((
#         np.squeeze(zcr),
#         np.squeeze(rmse),
#         np.ravel(mfcc_features)
#     ))
    
#     return result

# def load_model_and_predictors(model_json_path, model_weights_path, scaler_path, encoder_path):
#     """
#     Load the pre-trained model, scaler, and encoder
#     """
#     # Load model architecture
#     with open(model_json_path, 'r') as json_file:
#         loaded_model_json = json_file.read()
#     loaded_model = tf.keras.models.model_from_json(loaded_model_json)
    
#     # Load model weights
#     loaded_model.load_weights(model_weights_path)
    
#     # Compile model
#     loaded_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
#     # Load scaler
#     with open(scaler_path, "rb") as f:
#         scaler = pickle.load(f)
    
#     # Load encoder
#     with open(encoder_path, "rb") as f:
#         encoder = pickle.load(f)
    
#     return loaded_model, scaler, encoder

# def get_predict_feat(audio_data, sr, scaler, expected_shape=(1, 2376)):
#     """
#     Prepare features for prediction
#     """
#     res = extract_features(audio_data, sr)
    
#     # Ensure res is reshaped or padded to match the expected shape
#     if res.shape != expected_shape:
#         flat_size = np.prod(expected_shape)
#         if res.size < flat_size:
#             # Pad if the size is smaller than expected
#             pad_width = (0, flat_size - res.size)
#             res = np.pad(res, pad_width=pad_width, mode='constant')
#         else:
#             # Resize if the size is larger than expected
#             res = np.resize(res, expected_shape)

#     i_result = scaler.transform(res.reshape(1, -1))
#     final_result = np.expand_dims(i_result, axis=2)
#     return final_result

# def prediction(audio_data, sr, model, scaler, encoder):
#     """
#     Make a prediction using the loaded model
#     """
#     res = get_predict_feat(audio_data, sr, scaler)
#     predictions = model.predict(res)
    
#     # Get the label names
#     label_names = list(encoder.categories_[0])
    
#     # Get the index of the label with the highest confidence score
#     predicted_label_index = np.argmax(predictions)
    
#     # List to store confidence scores
#     confidence_scores = []
    
#     # Display predicted emotion and confidence for each label
#     print(f"\nPredicted Emotion: {label_names[predicted_label_index]}")
    
#     for label_index, label_name in enumerate(label_names):
#         confidence_score = predictions[0][label_index]
#         confidence_score = 0 if confidence_score < 0.001 else confidence_score
#         confidence_scores.append({'label': label_name, 'confidence': confidence_score})
    
#     sorted_confidence_scores = sorted(confidence_scores, key=lambda x: x['confidence'], reverse=True)
#     return sorted_confidence_scores

# def optimize_emotion_audio(emotion, model, scaler, encoder, iterations=20, sr=22050):
#     """
#     Optimize the audio generation to get high confidence for the target emotion
#     """
#     best_audio = None
#     best_confidence = 0
    
#     for i in range(iterations):
#         print(f"Iteration {i+1}/{iterations}")
        
#         # Generate audio with slightly different parameters each time
#         audio = generate_synthetic_audio(emotion, duration=2.5, sr=sr)
        
#         # Predict emotion
#         results = prediction(audio, sr, model, scaler, encoder)
        
#         # Check if this is the target emotion and has better confidence
#         for result in results:
#             if result['label'] == emotion:
#                 confidence = result['confidence']
#                 if confidence > best_confidence:
#                     best_confidence = confidence
#                     best_audio = audio
#                     print(f"New best confidence for {emotion}: {best_confidence:.4f}")
#                 break
    
#     if best_audio is not None:
#         print(f"Best confidence achieved for {emotion}: {best_confidence:.4f}")
#         return best_audio
#     else:
#         print(f"Could not optimize audio for {emotion}")
#         # Return the last generated audio as fallback
#         return audio

# # create_all_emotion_audio()
    
#     # 2. If you want to optimize audio for a specific emotion with your model:

#     # First load your model, scaler, and encoder
# model, scaler, encoder = load_model_and_predictors(
#     "results/CNN_model.json",
#     "results/best_model.weights.h5",
#     "results/scaler.pickle",
#     "results/encoder.pickle"
# )

# # Then optimize audio for a specific emotion
# emotion = "HAPPY"
# optimized_audio = optimize_emotion_audio(emotion, model, scaler, encoder, 1000)

# # Save the optimized audio
# sf.write(f"{emotion.lower()}_optimized.wav", optimized_audio, 22050)



In [11]:
#Check

def zcr(data,frame_length,hop_length):
    zcr=librosa.feature.zero_crossing_rate(data,frame_length=frame_length,hop_length=hop_length)
#     print(np.squeeze(zcr))
    return np.squeeze(zcr)

def rmse(data,frame_length=2048,hop_length=512):
    rmse=librosa.feature.rms(y=data,frame_length=frame_length,hop_length=hop_length)
    return np.squeeze(rmse)

def mfcc(data, sr, frame_length=2048, hop_length=512, flatten=True):
    mfcc_result = librosa.feature.mfcc(y=data, sr=sr, n_fft=frame_length, hop_length=hop_length)
    print(mfcc_result.shape)
    return np.squeeze(mfcc_result.T) if not flatten else np.ravel(mfcc_result.T)

def extract_features(data,sr=22050,frame_length=2048,hop_length=512):
    result=np.array([])

    result=np.hstack((
      result,
      zcr(data,frame_length,hop_length),
      rmse(data,frame_length,hop_length),
      mfcc(data,sr,frame_length,hop_length)
    ))

    return result

def get_predict_feat(path, expected_shape=(1, 2376)):
    d, s_rate = librosa.load(path, duration=2.5, offset=0.6)
    res = extract_features(d)

    # Ensure res is reshaped or padded to match the expected shape
    if res.shape != expected_shape:
        flat_size = np.prod(expected_shape)
        if res.size < flat_size:
            # Pad if the size is smaller than expected
            pad_width = (0, flat_size - res.size)
            res = np.pad(res, pad_width=pad_width, mode='constant')
        else:
            # Resize if the size is larger than expected
            res = np.resize(res, expected_shape)

    i_result = scaler.transform(res.reshape(1, -1))
    final_result = np.expand_dims(i_result, axis=2)

    return final_result

def prediction(path1, predicted_emo = []):
    print(path1)
    res = get_predict_feat(path1)
    predictions = loaded_model.predict(res)

    # Get the label names or define them if available
    label_names = list(encoder.categories_[0])

    # Get the index of the label with the highest confidence score
    predicted_label_index = np.argmax(predictions)

    # List to store confidence scores
    confidence_scores = []

    # Display predicted emotion and confidence for each label
    print(f"\nPredicted Emotion: {label_names[predicted_label_index]}")
    predicted_emo.append(label_names[predicted_label_index])
    for label_index, label_name in enumerate(label_names):
        confidence_score = predictions[0][label_index]
        confidence_score = 0 if confidence_score < 0.001 else confidence_score
        confidence_scores.append({'label': label_name, 'confidence': confidence_score})

    print("\n")

    sorted_confidence_scores = sorted(confidence_scores, key=lambda x: x['confidence'], reverse=True)

    return sorted_confidence_scores

In [15]:
# pathhh = "synthetic_audio.wav"

prediction("adversarial_disgust_genetic.wav")


adversarial_disgust_genetic.wav
(20, 82)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step

Predicted Emotion: angry




[{'label': 'angry', 'confidence': 0.9908312},
 {'label': 'fear', 'confidence': 0.008937717},
 {'label': 'disgust', 'confidence': 0},
 {'label': 'happy', 'confidence': 0},
 {'label': 'neutral', 'confidence': 0},
 {'label': 'sad', 'confidence': 0},
 {'label': 'surprise', 'confidence': 0}]

In [13]:
np.linspace(1, 1, 5)

array([1., 1., 1., 1., 1.])